# 개별종목 조합E — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합E 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합E의 피처 값만 지정합니다.
import json

COMBINATION = 'E'
FEATURE_COLUMNS = (
    'sector_ret_5',
    'sector_ret_20',
    'relative_ret_5_sector',
    'relative_ret_20_sector',
    'relative_ret_5_market',
    'sector_hv_20',
    'sector_beta_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 172058 162
조합E 피처: ('sector_ret_5', 'sector_ret_20', 'relative_ret_5_sector', 'relative_ret_20_sector', 'relative_ret_5_market', 'sector_hv_20', 'sector_beta_60')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3913,0.3701,0.0212,0.3694,0.2604,0.3296
1,2,balanced,999,20140414,20140711,0.4264,0.4743,-0.0479,0.3037,0.1611,0.2532
2,3,balanced,1248,20150421,20150716,0.3599,0.3301,0.0298,0.3562,0.3587,0.3583
3,4,balanced,1496,20160422,20160719,0.3596,0.4107,-0.0511,0.3349,0.1896,0.2717
4,5,balanced,1745,20170424,20170721,0.3738,0.4177,-0.0439,0.3161,0.2417,0.3007
5,6,balanced,1994,20180503,20180731,0.3990,0.3908,0.0082,0.3965,0.3089,0.3629
6,7,balanced,2243,20190514,20190806,0.3908,0.4612,-0.0704,0.3369,0.1232,0.2198
7,8,balanced,2492,20200518,20200807,0.3621,0.3144,0.0477,0.3540,0.3151,0.3424
8,9,balanced,2741,20210518,20210810,0.3897,0.4418,-0.0521,0.3635,0.2693,0.3322
9,10,balanced,2989,20220519,20220812,0.3483,0.3343,0.0140,0.3388,0.2990,0.3273


,OOS 폴드 평균
accuracy,0.3794
training_majority_baseline_accuracy,0.3846
accuracy_minus_training_majority_baseline,-0.0052
macro_f1,0.3503
down_recall,0.2577
core_harmonic_mean,0.3141


재실행 명령: python scripts/run_stock_model_experiment.py
